<a href="https://colab.research.google.com/github/arnaudm-cmd/C10---Projet-de-data-science/blob/main/Ex5_Analyse_visuelle_interactive_avec_Folium.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Launch Sites Locations Analysis with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [ ]:
!pip3 install folium
!pip3 install wget
!pip3 install pandas

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=9ff19f3f5a855b28da1304b644aa77a9dd63c09238f8ba9e79930e6c83385107
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [ ]:
import folium
import wget
import pandas as pd

In [ ]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [ ]:
# Télécharger le fichier CSV contenant les données géographiques
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
# Charger le fichier CSV dans un dataframe
spacex_df = pd.read_csv(spacex_csv_file)

Now, you can take a look at what are the coordinates for each site.


In [ ]:
# Sélectionner uniquement les colonnes utiles
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
# Grouper par site de lancement et garder la première ligne (pour éviter les doublons)
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
# Garder seulement les colonnes nécessaires
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
# Afficher les sites de lancement
print(launch_sites_df)

    Launch Site        Lat        Long
0   CCAFS LC-40  28.562302  -80.577356
1  CCAFS SLC-40  28.563197  -80.576820
2    KSC LC-39A  28.573255  -80.646895
3   VAFB SLC-4E  34.632834 -120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [ ]:
# Coordonnées du centre NASA Johnson Space Center à Houston
nasa_coordinate = [29.559684888503615, -95.0830971930759]
# Créer une carte centrée sur NASA avec zoom initial de 5
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [ ]:
# Créer un cercle bleu au centre NASA avec un popup
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Créer un marqueur avec le texte "NASA JSC"
marker = folium.map.Marker(
    nasa_coordinate,
    # Créer une icône personnalisée avec du texte
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
    )
)
# Ajouter le cercle et le marqueur à la carte
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


_TODO:_  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [ ]:
# Pour chaque site de lancement, ajouter un cercle et un marqueur
for index, record in launch_sites_df.iterrows():
    # Récupérer les coordonnées du site
    lat = record['Lat']
    lon = record['Long']
    site_name = record['Launch Site']

    # Créer un cercle pour le site de lancement
    circle = folium.Circle(
        [lat, lon],
        radius=1000,
        color='#0066cc',
        fill=True
    ).add_child(folium.Popup(site_name))

    # Créer un marqueur avec le nom du site
    marker = folium.map.Marker(
        [lat, lon],
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#0066cc;"><b>%s</b></div>' % site_name,
        )
    )

    # Ajouter le cercle et le marqueur à la carte
    site_map.add_child(circle)
    site_map.add_child(marker)

# Afficher la carte
site_map


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:
- Are all launch sites in proximity to the Equator line?
- Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


_TODO:_ Create a new column in `launch_sites` dataframe called `marker_color` to store the marker colors based on the `class` value


_TODO:_ For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [ ]:

# Créer une fonction pour assigner une couleur selon le résultat du lancement
def assign_marker_color(launch_outcome):
    # Si le lancement est réussi (class=1), retourner vert
    if launch_outcome == 1:
        return 'green'
    # Sinon, retourner rouge
    else:
        return 'red'

# Appliquer la fonction à chaque ligne pour créer une colonne marker_color
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)

# Afficher les dernières lignes avec la nouvelle colonne
print(spacex_df.tail(10))

# Créer une nouvelle carte pour les marqueurs de lancement
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Créer un cluster de marqueurs
marker_cluster = MarkerCluster()

# Pour chaque lancement dans le dataframe
for index, record in spacex_df.iterrows():
    # Récupérer les coordonnées et la couleur
    lat = record['Lat']
    lon = record['Long']
    color = record['marker_color']

    # Créer un marqueur avec la couleur appropriée
    marker = folium.Marker(
        location=[lat, lon],
        # Utiliser une icône avec la couleur du résultat
        icon=folium.Icon(color='white', icon_color=color)
    )

    # Ajouter le marqueur au cluster
    marker_cluster.add_child(marker)

# Ajouter le cluster à la carte
site_map.add_child(marker_cluster)

# Afficher la carte
site_map

     Launch Site        Lat       Long  class marker_color
46    KSC LC-39A  28.573255 -80.646895      1        green
47    KSC LC-39A  28.573255 -80.646895      1        green
48    KSC LC-39A  28.573255 -80.646895      1        green
49  CCAFS SLC-40  28.563197 -80.576820      1        green
50  CCAFS SLC-40  28.563197 -80.576820      1        green
51  CCAFS SLC-40  28.563197 -80.576820      0          red
52  CCAFS SLC-40  28.563197 -80.576820      0          red
53  CCAFS SLC-40  28.563197 -80.576820      0          red
54  CCAFS SLC-40  28.563197 -80.576820      1        green
55  CCAFS SLC-40  28.563197 -80.576820      0          red


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [ ]:
# Ajouter MousePosition pour voir les coordonnées de la souris
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

# Ajouter MousePosition à la carte
site_map.add_child(mouse_position)

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


In [ ]:
from math import sin, cos, sqrt, atan2, radians

# Fonction pour calculer la distance entre deux points (en km)
def calculate_distance(lat1, lon1, lat2, lon2):
    # Rayon approximatif de la Terre en km
    R = 6373.0

    # Convertir les degrés en radians
    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    # Calculer les différences
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    # Formule de Haversine
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    # Calculer la distance
    distance = R * c
    return distance

_TODO:_ Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [ ]:
# Coordonnées du site de lancement (exemple : CCAFS)
launch_site_lat = 28.5621
launch_site_lon = -80.5766

# Coordonnées de la côte la plus proche (trouvées avec MousePosition)
coastline_lat = 28.5600
coastline_lon = -80.5700

# Calculer la distance entre le site et la côte
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# Afficher la distance
print(f"Distance à la côte : {distance_coastline:.2f} km")

Distance à la côte : 0.69 km


_TODO:_ After obtained its coordinate, create a `folium.Marker` to show the distance


In [ ]:
# Créer un marqueur pour afficher la distance
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#ff6600;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
    )
)

# Ajouter le marqueur à la carte
site_map.add_child(distance_marker)

_TODO:_ Draw a `PolyLine` between a launch site to the selected coastline point


In [ ]:
# Créer une ligne (PolyLine) entre le site de lancement et la côte
# locations : liste des coordonnées [latitude, longitude] des deux points
# weight : épaisseur de la ligne
# color : couleur de la ligne
lines = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]],
    weight=2,
    color='red'
)

# Ajouter la ligne à la carte
site_map.add_child(lines)

# Afficher la carte
site_map

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


_TODO:_ Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [ ]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site
# Coordonnées de la ville la plus proche (trouvées avec MousePosition)
city_lat = 28.4162
city_lon = -80.4269

# Calculer la distance entre le site de lancement et la ville
distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)

# Afficher la distance
print(f"Distance à la ville : {distance_city:.2f} km")

# Créer un marqueur pour afficher la distance à la ville
distance_marker_city = folium.Marker(
    [city_lat, city_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#0066ff;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_city),
    )
)

# Ajouter le marqueur à la carte
site_map.add_child(distance_marker_city)

# Créer une ligne entre le site de lancement et la ville
lines_city = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [city_lat, city_lon]],
    weight=2,
    color='blue'
)

# Ajouter la ligne à la carte
site_map.add_child(lines_city)


Distance à la ville : 21.85 km


In [ ]:
# Coordonnées de la voie ferrée la plus proche (trouvées avec MousePosition)
railway_lat = 28.5400
railway_lon = -80.5400

# Calculer la distance entre le site de lancement et la voie ferrée
distance_railway = calculate_distance(launch_site_lat, launch_site_lon, railway_lat, railway_lon)

# Afficher la distance
print(f"Distance à la voie ferrée : {distance_railway:.2f} km")

# Créer un marqueur pour afficher la distance à la voie ferrée
distance_marker_railway = folium.Marker(
    [railway_lat, railway_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#ff00ff;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_railway),
    )
)

# Ajouter le marqueur à la carte
site_map.add_child(distance_marker_railway)

# Créer une ligne entre le site de lancement et la voie ferrée
lines_railway = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [railway_lat, railway_lon]],
    weight=2,
    color='purple'
)

# Ajouter la ligne à la carte
site_map.add_child(lines_railway)

Distance à la voie ferrée : 4.34 km


In [ ]:
# Coordonnées de la route la plus proche (trouvées avec MousePosition)
highway_lat = 28.5500
highway_lon = -80.5500

# Calculer la distance entre le site de lancement et la route
distance_highway = calculate_distance(launch_site_lat, launch_site_lon, highway_lat, highway_lon)

# Afficher la distance
print(f"Distance à la route : {distance_highway:.2f} km")

# Créer un marqueur pour afficher la distance à la route
distance_marker_highway = folium.Marker(
    [highway_lat, highway_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#00cc00;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_highway),
    )
)

# Ajouter le marqueur à la carte
site_map.add_child(distance_marker_highway)

# Créer une ligne entre le site de lancement et la route
lines_highway = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [highway_lat, highway_lon]],
    weight=2,
    color='green'
)

# Ajouter la ligne à la carte
site_map.add_child(lines_highway)

# Afficher la carte finale
site_map

Distance à la route : 2.93 km


After you plot distance lines to the proximities, you can answer the following questions easily:
- Are launch sites in close proximity to railways?
- Are launch sites in close proximity to highways?
- Are launch sites in close proximity to coastline?
- Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


Joseph Santarcangelo


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2021-05-26|1.0|Yan|Created the initial version|


Copyright © 2021 IBM Corporation. All rights reserved.
